# Sentinel matching script demo

Set one left/right Sentinel pair, then run the classical or checkpoint inference mode. Each mode writes a JSON result.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working/src/sentinel_matching').is_dir() else Path(r'D:/quantum_task')
os.chdir(PROJECT_ROOT)
OUTPUT_DIR = Path('/kaggle/working/sentinel_demo') if Path('/kaggle/working').is_dir() else PROJECT_ROOT / 'outputs/sentinel_demo'
LEFT_PATH = None
RIGHT_PATH = None
CHECKPOINT = None
INPUTS_READY = LEFT_PATH is not None and RIGHT_PATH is not None

def run_script(*args):
    completed = subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout.strip())

print(f'inputs_ready={INPUTS_READY}')
print('Set LEFT_PATH and RIGHT_PATH to two .SAFE scene directories before running inference.')

## Classical ORB

ORB detects keypoints, matches descriptors, and verifies matches with geometric RANSAC.

In [ ]:
if INPUTS_READY:
    run_script('python', 'src/sentinel_matching/inference/image_matcher_run.py', '--left', str(LEFT_PATH), '--right', str(RIGHT_PATH), '--method', 'orb', '--output-dir', str(OUTPUT_DIR), '--bands', 'B04,B03,B02', '--max-side', '1600')
    print((OUTPUT_DIR / 'orb_metrics.json').read_text(encoding='utf-8'))
else:
    print('ORB demo skipped: define LEFT_PATH and RIGHT_PATH.')

## Siamese checkpoint (optional)

A trained checkpoint returns one similarity score and a match decision for the same image pair.

In [ ]:
if INPUTS_READY and CHECKPOINT is not None:
    run_script('python', 'src/sentinel_matching/inference/image_matcher_run.py', '--left', str(LEFT_PATH), '--right', str(RIGHT_PATH), '--checkpoint', str(CHECKPOINT), '--output-dir', str(OUTPUT_DIR), '--bands', 'B04,B03,B02', '--device', 'auto')
    print((OUTPUT_DIR / 'checkpoint_metrics.json').read_text(encoding='utf-8'))
else:
    print('Checkpoint demo skipped: define LEFT_PATH, RIGHT_PATH, and CHECKPOINT.')

## Recorded benchmark result

Without raw scenes, this cell shows the saved Kaggle benchmark results for SIFT, ORB, and LoFTR.

In [ ]:
BENCHMARK = PROJECT_ROOT / 'src/sentinel_matching/data/kaggle_benchmark/all_methods_metrics.csv'
print(BENCHMARK.read_text(encoding='utf-8'))